# Assignment 25: RAG & LCEL Chains (Master Grand Final)
**Student:** Patan Khalilulla Khan

### Tasks 1-10 Implementation
This notebook demonstrates RAG, LCEL branching, and Pydantic structured output.

In [ ]:
# Task 1: Load, Split, Embed (FAISS)
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

loader = TextLoader('data/rag_data.txt')
docs = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=10)
chunks = text_splitter.split_documents(docs)
vectorstore = FAISS.from_documents(chunks, OpenAIEmbeddings())
retriever = vectorstore.as_retriever()
print(f"Loaded {len(chunks)} chunks.")

In [ ]:
# Task 6 & 8: Routing with RunnableBranch & RunnablePassthrough
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableBranch, RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-3.5-turbo")
prompt = ChatPromptTemplate.from_template("Context: {context}\nQuestion: {question}")

# RunnablePassthrough demo (Task 8)
rag_chain = (
    RunnableParallel({"context": retriever, "question": RunnablePassthrough()})
    | prompt
    | llm
    | StrOutputParser()
)

# Branching (Task 6)
branch = RunnableBranch(
    (lambda x: "rag" in x.lower(), rag_chain),
    ChatPromptTemplate.from_template("Answer directly: {question}") | llm | StrOutputParser()
)

print("Branches initialized.")

In [ ]:
# Task 9: Evaluation
print("Q1 (RAG):", branch.invoke("What is RAG?"))
print("Q2 (Direct):", branch.invoke("Hello, how are you?"))